In [11]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision.datasets import CIFAR10

In [12]:
# datasets & dataloader
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),(0.5, 0.5, 0.5))
])

trainset = CIFAR10(root="./data", train=True, download=True, transform=transforms)
testset = CIFAR10(root="./data", train=False, download=True, transform=transforms)

In [13]:
trainloader = DataLoader(trainset, batch_size=64, shuffle=True)
testloader = DataLoader(testset, batch_size=64)

# CNN

In [16]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128, 256),
            nn.ReLU(),

            nn.Linear(256, 10)
            
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layers(x)

        return x

In [17]:
model = CNN()

In [18]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

# Training the CNN

In [21]:
epochs = 10

for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images, labels in trainloader:
        optimizer.zero_grad()
        output = model.forward(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()

        epoch_training_loss += loss.item()

    print(f"epoch value: {epoch+1}/{epochs} and loss = {epoch_training_loss/len(trainloader)}")

epoch value: 1/10 and loss = 1.0542274154817965
epoch value: 2/10 and loss = 0.8196638332837073
epoch value: 3/10 and loss = 0.6811376689263927
epoch value: 4/10 and loss = 0.5706577528163296
epoch value: 5/10 and loss = 0.476070775476563
epoch value: 6/10 and loss = 0.3854550645612847
epoch value: 7/10 and loss = 0.3077779185703343
epoch value: 8/10 and loss = 0.24250731292797628
epoch value: 9/10 and loss = 0.18692165105353536
epoch value: 10/10 and loss = 0.15529023854971846


In [22]:
# Evaluate CNN

correct_labels =0
total_labels = 0

model.eval()

with torch.no_grad():
    for images, labels in testloader:
        outputs = model.forward(images)
        _, predicted = torch.max(outputs, 1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

print(f"accuracy: {correct_labels / total_labels * 100} * 100")

accuracy: 0.758 * 100
